In [1]:
import os, sys
import subprocess
import pandas
import pandas as pd
from utilities import *
import pyrosetta
import csv
from pyrosetta import init, pose_from_file
from pyrosetta.rosetta.core.select.residue_selector import ChainSelector
from pyrosetta.rosetta.core.pack.task import TaskFactory
from pyrosetta.rosetta.core.pack.task.operation import InitializeFromCommandline, RestrictResidueToRepacking
from pyrosetta.rosetta.protocols.task_operations import RestrictChainToRepackingOperation
from pyrosetta.rosetta.core.pack.task.operation import RestrictToRepacking
from pyrosetta.rosetta.protocols.backrub import BackrubMover
from pyrosetta.rosetta.core.select.movemap import MoveMapFactory
from pyrosetta.rosetta.protocols.simple_filters import ContactMolecularSurfaceFilter
from pyrosetta.rosetta.core.scoring import CA_rmsd
import time

In [2]:
pyrosetta.init(
    extra_options="-relax:default_repeats 1 -ignore_zero_occupancy false -multithreading:total_threads 1 -constant_seed false"
)

PyRosetta-4 2023 [Rosetta PyRosetta4.conda.linux.cxx11thread.serialization.CentOS.python311.Release 2023.36+release.2660f2773c32e732ce07c88dc75b39e18728f3c0 2023-09-06T18:10:05] retrieved from: http://www.pyrosetta.org
(C) Copyright Rosetta Commons Member Institutions. Created in JHU by Sergey Lyskov and PyRosetta Team.
core.init: Checking for fconfig files in pwd and ./rosetta/flags
core.init: Rosetta version: PyRosetta4.conda.linux.cxx11thread.serialization.CentOS.python311.Release r357 2023.36+release.2660f27 2660f2773c32e732ce07c88dc75b39e18728f3c0 http://www.pyrosetta.org 2023-09-06T18:10:05
core.init: command: PyRosetta -ex1 -ex2aro -relax:default_repeats 1 -ignore_zero_occupancy false -multithreading:total_threads 1 -constant_seed false -database /ifs/scratch/public_softwares/conda_env/newpyrosetta/lib/python3.11/site-packages/pyrosetta/database
basic.random.init_random_generator: 'RNG device' seed mode, using '/dev/urandom', seed=1267241408 seed_offset=0 real_seed=1267241408 th

In [3]:

def perform_chainA_backrub(pdb_list: list, backrub_output: os.path, n_struct_backrub=10, n_trials_backrub=1000,chain_res_design_dict={}):
	print(f"[INFO.perform_chainA_backrub] PERFORMING perform_chainA_backrub IN ENERGY METHODS")
	#init()
	make_dir(backrub_output)
	final: list = []
	index = 0
	print(f"[INFO.perform_chainA_backrub] chain_res_design_dict {chain_res_design_dict}")

	for pdb_path in pdb_list:
		file = pdb_path.split("/")[-1][0:-4]
		print(f"[INFO.perform_chainA_backrub] PDB PATH: {pdb_path}")
		# print(f"[INFO.perform_chainA_backrub] FILE: {file}")

		working_pose: pyrosetta.Pose = pyrosetta.pose_from_file(pdb_path)
		info: pyrosetta.rosetta.core.pose.PDBInfo = working_pose.pdb_info()
		pose_list = []
		for chain_id, res_list in chain_res_design_dict.items():
			for res in res_list:
				pose_list.append(str(info.pdb2pose(chain_id, res)))

		print(f"[INFO.perform_chainA_backrub] Pose list: {pose_list}")
		pose_list_str = ",".join(pose_list)
		
		hsa_res_selector= pyrosetta.rosetta.core.select.residue_selector.ResidueIndexSelector(pose_list_str)

		nbr_selector = pyrosetta.rosetta.core.select.residue_selector.NeighborhoodResidueSelector()
		nbr_selector.set_focus_selector(hsa_res_selector)
		nbr_selector.set_distance(10.0)

		mmf = pyrosetta.rosetta.core.select.movemap.MoveMapFactory()
		
		mmf.add_bb_action(pyrosetta.rosetta.core.select.movemap.move_map_action.mm_enable, hsa_res_selector)
		mmf.add_bb_action(pyrosetta.rosetta.core.select.movemap.move_map_action.mm_enable, nbr_selector)

		br_mover = pyrosetta.rosetta.protocols.backrub.BackrubMover()
		br_mover.set_movemap_factory(mmf)

		# PRIME THE MOVER
		print(f"[INFO.perform_chainA_backrub] Priming backrub mover (defaults to all residues)...")
		temp_pose = pyrosetta.Pose(working_pose)
		br_mover.apply(temp_pose)

		print(f"[INFO.perform_chainA_backrub] Clearing segments...")
		br_mover.clear_segments()

		from pyrosetta.rosetta.core.id import AtomID

		subset = hsa_res_selector.apply(working_pose)
		resnums = pyrosetta.rosetta.core.select.get_residues_from_subset(subset)

		for i in resnums:
			if i > 1 and i < working_pose.total_residue():
				ca1 = working_pose.residue(i-1).atom_index("CA")
				ca2 = working_pose.residue(i+1).atom_index("CA")

				br_mover.add_segment(
					AtomID(ca1, i-1),
					AtomID(ca2, i+1)
				)

		print(f"[INFO.perform_chainA_backrub] Num segments from neighborhood selector: {br_mover.num_segments()}", flush=True)
		# br_protocol = pyrosetta.rosetta.protocols.backrub.BackrubProtocol()
		# print(f"[INFO.perform_chainA_backrub] backrub mover type: {type(br_mover)}")

		scorefxn = pyrosetta.get_fa_scorefxn()
		score_input = scorefxn(working_pose)

		print(f"[INFO.perform_chainA_backrub] Num backrub structures to output: {n_struct_backrub}")
		for i in range(n_struct_backrub):

			pose_copy = pyrosetta.Pose(working_pose)

			# # Apply a single backrub move (minimal change)
			# br_mover.apply(pose_copy)
			# print(f"[INFO.perform_chainA_backrub] Applied backrub mover")

			# br_protocol = pyrosetta.rosetta.protocols.backrub.BackrubProtocol() # is a JD2 application wrapper
			# br_protocol.set_backrub_mover(br_mover)
			# br_protocol.set_scorefunction(scorefxn)
			# br_protocol.set_options(ntrials=1000)
			
			# # Restrict pivots
			# subset = hsa_res_selector.apply(pose_copy)
			# br_protocol.set_pivots_from_residue_subset(subset)
			# print(f"[INFO.perform_chainA_backrub] Applying backrub protocol")
			# br_protocol.apply(pose_copy)

			# --- Prove raw backrub actually moves atoms before MC ---
			pose0 = pyrosetta.Pose(pose_copy)

			for _ in range(5):
				br_mover.apply(pose_copy)

			first_rmsd = CA_rmsd(pose0, pose_copy)
			print(f"[INFO.perform_chainA_backrub] first rmsd {first_rmsd}")

			if first_rmsd < 5e-3:
				raise RuntimeError(
					f"[INFO.perform_chainA_backrub] Backrub proposals are effectively zero "
					f"(ΔRMSD after 5 raw moves = {first_rmsd:.4f} Å). "
					"BB DOFs likely blocked or pivots invalid.", flush=True
				)

			# reset pose before MC
			pose_copy.assign(pose0)

			mc = pyrosetta.rosetta.protocols.moves.MonteCarlo(
				pose_copy,
				scorefxn,
				1.5  # temperature
			)

			accepted = 0
			start = time.time()
			
			print(f"[INFO.perform_chainA_backrub] Starting MC", flush=True)
			for step in range(n_trials_backrub):  # ntrials
				br_mover.apply(pose_copy)
				if mc.boltzmann(pose_copy):
					accepted += 1
				if step % 500 == 0:
					print(f"[INFO.perform_chainA_backrub] MC step {step}", flush=True)

			end = time.time()

			# mc.recover_low(pose_copy)

			final_rmsd = CA_rmsd(pose0, pose_copy)
			acc_rate = accepted / 1000.0

			print(
				f"[INFO.perform_chainA_backrub] accepted={accepted}/{n_trials_backrub} ({acc_rate*100:4.1f}%)  "
				f"ΔRMSD={final_rmsd:.3f} Å  "
				f"MC loop time: {end - start:.2f}s",
				flush=True
			)
			delta_score = scorefxn(pose_copy) - score_input
			print(f"[INFO.perform_chainA_backrub] delta score (backrub structure {i}) = {delta_score}")

			output_path = os.path.join(
				backrub_output,
				str(file) + "_" + str(index) + "_br.pdb"
			)

			final.append(output_path)
			pose_copy.dump_pdb(output_path)
			print(f"[INFO.perform_chainA_backrub] Dump pdb to output path: {output_path}", flush=True)
			index += 1
	
	return final



In [4]:
pdb_list = ['/ifs/share/TF_Project/Design/TF_project_design_step/GalS/Design_input/gals_design_input.pdb']
backrub_output = '/ifs/scratch/home/bs3281/TF_Project_BS/Test_notebook'
chain_res_design_dict={'A': [275, 276, 277, 278, 279, 280, 281], 'B': [275, 276, 277, 278, 279, 280, 281]}

In [5]:
final = perform_chainA_backrub(pdb_list, backrub_output, n_struct_backrub=4, n_trials_backrub=100, chain_res_design_dict=chain_res_design_dict)

[INFO.perform_chainA_backrub] PERFORMING perform_chainA_backrub IN ENERGY METHODS
[INFO] Directory /ifs/scratch/home/bs3281/TF_Project_BS/Test_notebook already exists
[INFO.perform_chainA_backrub] chain_res_design_dict {'A': [275, 276, 277, 278, 279, 280, 281], 'B': [275, 276, 277, 278, 279, 280, 281]}
[INFO.perform_chainA_backrub] PDB PATH: /ifs/share/TF_Project/Design/TF_project_design_step/GalS/Design_input/gals_design_input.pdb
core.chemical.GlobalResidueTypeSet: Finished initializing fa_standard residue type set.  Created 985 residue types
core.chemical.GlobalResidueTypeSet: Total time to initialize 0.962018 seconds.
core.import_pose.import_pose: File '/ifs/share/TF_Project/Design/TF_project_design_step/GalS/Design_input/gals_design_input.pdb' automatically determined to be of type PDB
core.io.pose_from_sfr.PoseFromSFRBuilder: [ WARNING ] discarding 1 atoms at position 87 in file /ifs/share/TF_Project/Design/TF_project_design_step/GalS/Design_input/gals_design_input.pdb. Best matc

In [6]:
import pyrosetta
pyrosetta.init()

br0 = '/ifs/scratch/home/bs3281/TF_Project_BS/Test_notebook/gals_design_input_0_br.pdb'
br1 = '/ifs/scratch/home/bs3281/TF_Project_BS/Test_notebook/gals_design_input_1_br.pdb'

pose0 = pyrosetta.pose_from_file(br0)
pose1 = pyrosetta.pose_from_file(br1)

scorefxn = pyrosetta.get_fa_scorefxn()

score0 = scorefxn(pose0)
score1 = scorefxn(pose1)

print(f"pose0 total score: {score0:.3f}")
print(f"pose1 total score: {score1:.3f}")

rmsd= CA_rmsd(pose0, pose1)
print(f"rmsd = {rmsd}")

PyRosetta-4 2023 [Rosetta PyRosetta4.conda.linux.cxx11thread.serialization.CentOS.python311.Release 2023.36+release.2660f2773c32e732ce07c88dc75b39e18728f3c0 2023-09-06T18:10:05] retrieved from: http://www.pyrosetta.org
(C) Copyright Rosetta Commons Member Institutions. Created in JHU by Sergey Lyskov and PyRosetta Team.
core.init: Checking for fconfig files in pwd and ./rosetta/flags


core.init: Rosetta version: PyRosetta4.conda.linux.cxx11thread.serialization.CentOS.python311.Release r357 2023.36+release.2660f27 2660f2773c32e732ce07c88dc75b39e18728f3c0 http://www.pyrosetta.org 2023-09-06T18:10:05
core.init: command: PyRosetta -ex1 -ex2aro -database /ifs/scratch/public_softwares/conda_env/newpyrosetta/lib/python3.11/site-packages/pyrosetta/database
basic.random.init_random_generator: 'RNG device' seed mode, using '/dev/urandom', seed=-1505581675 seed_offset=0 real_seed=-1505581675 thread_index=0
basic.random.init_random_generator: RandomGenerator:init: Normal mode, seed=-1505581675 RG_type=mt19937
core.import_pose.import_pose: File '/ifs/scratch/home/bs3281/TF_Project_BS/Test_notebook/gals_design_input_0_br.pdb' automatically determined to be of type PDB
core.conformation.Conformation: Found disulfide between residues 156 317
core.import_pose.import_pose: File '/ifs/scratch/home/bs3281/TF_Project_BS/Test_notebook/gals_design_input_1_br.pdb' automatically determined 

In [9]:
from pathlib import Path
protein_mpnn_repo = Path("/ifs/scratch/home/bs3281/Parrots_BS/ProteinMPNN")
protein_mpnn_parent = protein_mpnn_repo.parent

if str(protein_mpnn_parent) not in sys.path:
    sys.path.insert(0, str(protein_mpnn_parent))

from utilities import *
import energy_methods_original
from ProteinMPNN import protein_mpnn_run

In [10]:
from energy_methods_original import design_round_for_WT

pre_step1 = final
output_mpnn = '/ifs/scratch/home/bs3281/TF_Project_BS/Test_notebook/step1_mpnn'
temp_dir = '/ifs/scratch/home/bs3281/TF_Project_BS/Test_notebook/temp'

step_1_passed = design_round_for_WT(
	pre_step1,
	output_mpnn,
	True,
	False,
	True,
	1,
	2,
	1,
	False,
	True,
	chain_res_design_dict,
	temp_dir=temp_dir,
)

[INFO.design_round_for_WT] Starting design round...
[INFO] Directory /ifs/scratch/home/bs3281/TF_Project_BS/Test_notebook/step1_mpnn already exists
core.import_pose.import_pose: File '/ifs/scratch/home/bs3281/TF_Project_BS/Test_notebook/gals_design_input_0_br.pdb' automatically determined to be of type PDB
core.conformation.Conformation: Found disulfide between residues 156 317
mutable list
[['275', '276', '277', '278', '279', '280', '281'], ['275', '276', '277', '278', '279', '280', '281']]
Pose list ['275', '276', '277', '278', '279', '280', '281', '620', '621', '622', '623', '624', '625', '626']
275,276,277,278,279,280,281,620,621,622,623,624,625,626
Res selector: vector1_unsigned_long[275, 276, 277, 278, 279, 280, 281, 620, 621, 622, 623, 624, 625, 626]
core.import_pose.import_pose: File '/ifs/scratch/home/bs3281/TF_Project_BS/Test_notebook/gals_design_input_0_br.pdb' automatically determined to be of type PDB
core.conformation.Conformation: Found disulfide between residues 156 317

usage: ipykernel_launcher.py [-h] [--input_path INPUT_PATH]
                             [--output_path OUTPUT_PATH] [--ca_only]
ipykernel_launcher.py: error: unrecognized arguments: --f=/run/user/1003/jupyter/runtime/kernel-v369bf878ade14e1401cd6d0d99fa39ed73d868c37.json


SystemExit: 2

/ifs/scratch/public_softwares/conda_env/newpyrosetta/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3534: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
